# Add no-snore class

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv(r"C:\V89\Snore_Apnea_Analyze\EDF_RML\data_csv\use_train\train\use_train_osa.csv")
df['type'].value_counts()

type
ObstructiveApnea    645
Snore               279
Name: count, dtype: int64

In [4]:
df

,patient_id,event_id,family,type,start_sec,duration_sec,end_sec,segment_index,segment_local_start_sec,recording_start_iso
0,995,131,Nasal,Snore,3934.5,3.5,3938.0,1,334.5,2019-04-17T22:35:00
1,995,135,Nasal,Snore,4053.5,9.5,4063.0,1,453.5,2019-04-17T22:35:00
2,995,137,Nasal,Snore,4107.0,4.5,4111.5,1,507.0,2019-04-17T22:35:00
3,995,138,Nasal,Snore,4113.5,9.5,4123.0,1,513.5,2019-04-17T22:35:00
4,995,140,Respiratory,ObstructiveApnea,4242.0,10.0,4252.0,1,642.0,2019-04-17T22:35:00
...,...,...,...,...,...,...,...,...,...,...
919,1089,1517,Respiratory,ObstructiveApnea,15161.5,28.5,15190.0,4,761.5,2019-07-04T22:04:42
920,1089,1533,Nasal,Snore,15301.5,4.5,15306.0,4,901.5,2019-07-04T22:04:42
921,1089,1552,Nasal,Snore,15487.5,6.0,15493.5,4,1087.5,2019-07-04T22:04:42
922,1089,1558,Respiratory,ObstructiveApnea,15553.0,23.5,15576.5,4,1153.0,2019-07-04T22:04:42


In [9]:
df['duration_sec'].mean()

np.float64(15.365800865800866)

In [12]:
import pandas as pd
import numpy as np

# ==== Load your existing CSV ====
csv_path = r"C:\V89\Snore_Apnea_Analyze\EDF_RML\data_csv\use_train\train\use_train_osa.csv"
df = pd.read_csv(csv_path)

# Sort by patient and start time
df = df.sort_values(["patient_id", "start_sec"]).reset_index(drop=True)

results = []

# Loop each patient
for pid, group in df.groupby("patient_id"):
    # occupied ranges (start_sec, end_sec)
    occupied = group[["start_sec", "end_sec"]].values.tolist()
    occupied.sort()
    
    # total recording duration (max end_sec seen in this patient)
    total_time = group["end_sec"].max()
    
    # scan in 30s steps
    t = 0
    while t + 30 <= total_time:
        window = (t, t+30)
        overlap = any(not (window[1] <= s or window[0] >= e) for s,e in occupied)
        if not overlap:
            results.append([pid, t, t+30])
        t += 30  # slide by 30s non-overlapping windows
    
    print(f"Patient {pid}: {len(results)} possible no_Snore windows")

# Show summary
summary = pd.DataFrame(results, columns=["patient_id", "start_sec", "end_sec"])
print(summary.groupby("patient_id").size())


Patient 995: 470 possible no_Snore windows
Patient 999: 785 possible no_Snore windows
Patient 1000: 1114 possible no_Snore windows
Patient 1006: 1327 possible no_Snore windows
Patient 1008: 1703 possible no_Snore windows
Patient 1089: 1970 possible no_Snore windows
patient_id
995     470
999     315
1000    329
1006    213
1008    376
1089    267
dtype: int64


In [ ]:
import pandas as pd
import numpy as np

# ==== Load original events ====
csv_path = r"C:\V89\Snore_Apnea_Analyze\EDF_RML\data_csv\use_train\train\use_train_osa.csv"
df = pd.read_csv(csv_path)

# Sort
df = df.sort_values(["patient_id", "start_sec"]).reset_index(drop=True)

no_snore_rows = []

# ==== Generate no_Snore for each patient ====
for pid, group in df.groupby("patient_id"):
    occupied = group[["start_sec", "end_sec"]].values.tolist()
    occupied.sort()
    total_time = group["end_sec"].max()
    
    # collect free windows
    free_windows = []
    t = 0
    while t + 30 <= total_time:
        window = (t, t+30)
        overlap = any(not (window[1] <= s or window[0] >= e) for s,e in occupied)
        if not overlap:
            free_windows.append(window)
        t += 30  # step by 30s
    
    # randomly sample 40 windows
    np.random.seed(42)  # for reproducibility
    chosen = np.random.choice(len(free_windows), size=40, replace=False)
    
    for idx, win in enumerate([free_windows[i] for i in chosen]):
        start, end = win
        no_snore_rows.append({
            "patient_id": pid,
            "event_id": f"ns_{pid}_{idx}",
            "family": "Respiratory",
            "type": "no_Snore",
            "start_sec": start,
            "duration_sec": 30,
            "end_sec": end,
            "segment_index": 0,
            "segment_local_start_sec": start,
            "recording_start_iso": group["recording_start_iso"].iloc[0]
        })

# ==== Combine with original data ====
df_no_snore = pd.DataFrame(no_snore_rows)
df_final = pd.concat([df, df_no_snore], ignore_index=True)

# ==== Save new CSV ====
out_path = r"C:\V89\Snore_Apnea_Analyze\EDF_RML\data_csv\use_train\train\use_train_ose_normal.csv"
df_final.to_csv(out_path, index=False)

print("✅ Finished! Saved to:", out_path)
print("New class distribution:")
print(df_final["type"].value_counts())


✅ Finished! Saved to: C:\V89\Snore_Apnea_Analyze\EDF_RML\data_csv\use_train\train\use_train_ose_normal.csv
New class distribution:
type
ObstructiveApnea    645
Snore               279
no_Snore            240
Name: count, dtype: int64


#  Do the same with test class

In [15]:
import pandas as pd
import numpy as np

# ==== Load your existing CSV ====
csv_path = r"C:\V89\Snore_Apnea_Analyze\EDF_RML\data_csv\use_test\test\use_test_osa.csv"
df = pd.read_csv(csv_path)

# Sort by patient and start time
df = df.sort_values(["patient_id", "start_sec"]).reset_index(drop=True)

results = []

# Loop each patient
for pid, group in df.groupby("patient_id"):
    # occupied ranges (start_sec, end_sec)
    occupied = group[["start_sec", "end_sec"]].values.tolist()
    occupied.sort()
    
    # total recording duration (max end_sec seen in this patient)
    total_time = group["end_sec"].max()
    
    # scan in 30s steps
    t = 0
    while t + 30 <= total_time:
        window = (t, t+30)
        overlap = any(not (window[1] <= s or window[0] >= e) for s,e in occupied)
        if not overlap:
            results.append([pid, t, t+30])
        t += 30  # slide by 30s non-overlapping windows
    
    print(f"Patient {pid}: {len(results)} possible no_Snore windows")

# Show summary
summary = pd.DataFrame(results, columns=["patient_id", "start_sec", "end_sec"])
print(summary.groupby("patient_id").size())


Patient 1010: 76 possible no_Snore windows
Patient 1014: 169 possible no_Snore windows
Patient 1016: 706 possible no_Snore windows
patient_id
1010     76
1014     93
1016    537
dtype: int64


In [16]:
import pandas as pd
import numpy as np

# ==== Load original events ====
csv_path = r"C:\V89\Snore_Apnea_Analyze\EDF_RML\data_csv\use_test\test\use_test_osa.csv"
df = pd.read_csv(csv_path)

# Sort
df = df.sort_values(["patient_id", "start_sec"]).reset_index(drop=True)

no_snore_rows = []

# ==== Generate no_Snore for each patient ====
for pid, group in df.groupby("patient_id"):
    occupied = group[["start_sec", "end_sec"]].values.tolist()
    occupied.sort()
    total_time = group["end_sec"].max()
    
    # collect free windows
    free_windows = []
    t = 0
    while t + 30 <= total_time:
        window = (t, t+30)
        overlap = any(not (window[1] <= s or window[0] >= e) for s,e in occupied)
        if not overlap:
            free_windows.append(window)
        t += 30  # step by 30s
    
    # randomly sample 40 windows
    np.random.seed(42)  # for reproducibility
    chosen = np.random.choice(len(free_windows), size=40, replace=False)
    
    for idx, win in enumerate([free_windows[i] for i in chosen]):
        start, end = win
        no_snore_rows.append({
            "patient_id": pid,
            "event_id": f"ns_{pid}_{idx}",
            "family": "Respiratory",
            "type": "no_Snore",
            "start_sec": start,
            "duration_sec": 30,
            "end_sec": end,
            "segment_index": 0,
            "segment_local_start_sec": start,
            "recording_start_iso": group["recording_start_iso"].iloc[0]
        })

# ==== Combine with original data ====
df_no_snore = pd.DataFrame(no_snore_rows)
df_final = pd.concat([df, df_no_snore], ignore_index=True)

# ==== Save new CSV ====
out_path = r"C:\V89\Snore_Apnea_Analyze\EDF_RML\data_csv\use_test\test\use_test_ose_normal.csv"
df_final.to_csv(out_path, index=False)

print("✅ Finished! Saved to:", out_path)
print("New class distribution:")
print(df_final["type"].value_counts())


✅ Finished! Saved to: C:\V89\Snore_Apnea_Analyze\EDF_RML\data_csv\use_test\test\use_test_ose_normal.csv
New class distribution:
type
ObstructiveApnea    447
Snore               415
no_Snore            120
Name: count, dtype: int64
